# JobSpy Job Scraper

This notebook uses [JobSpy](https://github.com/speedyapply/JobSpy) to scrape jobs from LinkedIn, Indeed, Glassdoor, Google, ZipRecruiter & more.

**Usage:** Modify the search parameters below and run all cells to fetch jobs.

In [1]:
# Install JobSpy if not already installed
%pip install -U python-jobspy -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from jobspy import scrape_jobs
import json
from IPython.display import display, HTML, JSON
from datetime import datetime

## Search Parameters

Customize these parameters to search for jobs:

In [11]:
# Search configuration
SEARCH_TERM = "data analyst"  # Job title/keywords
LOCATION = "Remote"  # Location (e.g., "San Francisco, CA", "Remote", "India")
SITES = ["indeed", "linkedin", "zip_recruiter", "google", "naukri"]  # Job boards to search
RESULTS_WANTED = 50  # Number of results per site
HOURS_OLD = 72  # Only jobs posted in last N hours
COUNTRY = "india"  # For Indeed/Glassdoor (USA, India, UK, etc.)
IS_REMOTE = True  # Filter for remote jobs only

## Scrape Jobs

This will fetch jobs from the specified job boards:

In [12]:
print(f"🔍 Searching for '{SEARCH_TERM}' jobs in '{LOCATION}'...")
print(f"📊 Sites: {', '.join(SITES)}")
print(f"⏰ Jobs posted in last {HOURS_OLD} hours")
print("\n⏳ This may take 1-3 minutes...\n")

try:
    jobs_df = scrape_jobs(
        site_name=SITES,
        search_term=SEARCH_TERM,
        location=LOCATION,
        results_wanted=RESULTS_WANTED,
        hours_old=HOURS_OLD,
        country_indeed=COUNTRY,
        is_remote=IS_REMOTE,
        verbose=1  # 0=errors only, 1=errors+warnings, 2=all logs
    )
    
    print(f"\n✅ Found {len(jobs_df)} jobs!")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    jobs_df = pd.DataFrame()

🔍 Searching for 'data analyst' jobs in 'Remote'...
📊 Sites: indeed, linkedin, zip_recruiter, google, naukri
⏰ Jobs posted in last 72 hours

⏳ This may take 1-3 minutes...



2026-06-25 15:26:01,978 - ERROR - JobSpy:ZipRecruiter - ZipRecruiter response status code 403 with response: {"error_code":"forbidden aa","error_message":"forbidden aa","request_id":"CFRAY:a11319850bbe3e0f-IAD","status_code":403}
2026-06-25 15:26:02,058 - ERROR - JobSpy:Naukri - Naukri API response status code 406 - {"message":"recaptcha required","statusCode":406,"validationErrors":[]}
2026-06-25 15:26:04,797 - WARNING - JobSpy:Google - initial cursor not found, try changing your query or there was at most 10 results



✅ Found 66 jobs!


## Results Summary

In [13]:
if len(jobs_df) > 0:
    print(f"\n📈 Summary:")
    print(f"   Total jobs: {len(jobs_df)}")
    
    if 'site' in jobs_df.columns:
        print(f"\n📊 Jobs by source:")
        print(jobs_df['site'].value_counts().to_string())
    
    if 'job_type' in jobs_df.columns:
        print(f"\n💼 Job types:")
        print(jobs_df['job_type'].value_counts().to_string())
    
    if 'is_remote' in jobs_df.columns:
        remote_count = jobs_df['is_remote'].sum() if jobs_df['is_remote'].dtype == bool else 0
        print(f"\n🏠 Remote jobs: {remote_count}")
else:
    print("\n⚠️ No jobs found. Try adjusting your search parameters.")


📈 Summary:
   Total jobs: 66

📊 Jobs by source:
site
linkedin    50
indeed      16

💼 Job types:
job_type
fulltime                6
parttime, internship    1
parttime                1
fulltime, internship    1

🏠 Remote jobs: 26


## Job Listings Table

In [14]:
if len(jobs_df) > 0:
    # Select key columns for display
    display_cols = ['title', 'company', 'location', 'site', 'job_type', 'job_url']
    if 'min_amount' in jobs_df.columns and 'max_amount' in jobs_df.columns:
        display_cols.extend(['min_amount', 'max_amount'])
    if 'date_posted' in jobs_df.columns:
        display_cols.append('date_posted')
    
    # Filter to available columns
    display_cols = [col for col in display_cols if col in jobs_df.columns]
    
    # Display table
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 50)
    pd.set_option('display.width', None)
    
    display(jobs_df[display_cols].head(100))
else:
    print("No jobs to display.")

,title,company,location,site,job_type,job_url,min_amount,max_amount,date_posted
0,Data Analyst,Sun Pharmaceuticals laboratories Ltd,"Remote, IN",indeed,"parttime, internship",https://in.indeed.com/viewjob?jk=1954446c79d9c3ae,None,None,2026-06-24
1,Data Analyst I - AMS,Nordson,"Remote, IN",indeed,fulltime,https://in.indeed.com/viewjob?jk=6222af7849c97756,None,None,2026-06-24
2,International Risk Analyst,Vanguardvisor,"Remote, IN",indeed,parttime,https://in.indeed.com/viewjob?jk=68c417e1d14f74f1,None,None,2026-06-24
3,Digital Marketing Analyst,IT WEB SERVICES,"Remote, IN",indeed,fulltime,https://in.indeed.com/viewjob?jk=71eea98586fe6487,None,None,2026-06-24
4,Senior Analyst,Jabil,"Remote, IN",indeed,fulltime,https://in.indeed.com/viewjob?jk=f9cd62402cc86277,None,None,2026-06-24
...,...,...,...,...,...,...,...,...,...
61,Data Analyst Intern,NuvoAir Medical,,linkedin,NaN,https://www.linkedin.com/jobs/view/4433002236,None,None,NaN
62,Remote Data Analyst,Turing,"Mumbai, Maharashtra, India",linkedin,NaN,https://www.linkedin.com/jobs/view/4432829338,None,None,NaN
63,Data Analyst,SGF Global,,linkedin,NaN,https://www.linkedin.com/jobs/view/4432568269,None,None,NaN
64,Data Analyst,"PlanIT Group, LLC",,linkedin,NaN,https://www.linkedin.com/jobs/view/4433059001,None,None,NaN


## Export Results

In [15]:
if len(jobs_df) > 0:
    # Export to CSV
    csv_filename = f"jobs_{SEARCH_TERM.replace(' ', '_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    jobs_df.to_csv(csv_filename, index=False, encoding='utf-8')
    print(f"✅ Exported to: {csv_filename}")
    
    # Export to JSON (for web integration)
    json_filename = f"jobs_{SEARCH_TERM.replace(' ', '_')}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    jobs_df.to_json(json_filename, orient='records', indent=2, date_format='iso')
    print(f"✅ Exported to: {json_filename}")
    
    # Display download link
    display(HTML(f"<p><a href='{csv_filename}' download>📥 Download CSV</a> | <a href='{json_filename}' download>📥 Download JSON</a></p>"))
else:
    print("No jobs to export.")

✅ Exported to: jobs_data_analyst_20260625_152801.csv
✅ Exported to: jobs_data_analyst_20260625_152801.json


## JSON Output (for API integration)

The jobs data as JSON:

In [16]:
if len(jobs_df) > 0:
    jobs_json = jobs_df.to_dict(orient='records')
    # Clean up the JSON (remove NaN values)
    import math
    def clean_json(obj):
        if isinstance(obj, dict):
            return {k: clean_json(v) for k, v in obj.items() if not (isinstance(v, float) and math.isnan(v))}
        elif isinstance(obj, list):
            return [clean_json(item) for item in obj]
        return obj
    
    clean_jobs = clean_json(jobs_json)
    print(f"📋 JSON output ({len(clean_jobs)} jobs):")
    print(json.dumps(clean_jobs[:5], indent=2, default=str))  # Show first 5 as preview
    print(f"\n... ({len(clean_jobs) - 5} more jobs)")
else:
    print("No jobs to display.")

📋 JSON output (66 jobs):
[
  {
    "id": "in-1954446c79d9c3ae",
    "site": "indeed",
    "job_url": "https://in.indeed.com/viewjob?jk=1954446c79d9c3ae",
    "job_url_direct": "http://in.indeed.com/job/data-analyst-1954446c79d9c3ae",
    "title": "Data Analyst",
    "company": "Sun Pharmaceuticals laboratories Ltd",
    "location": "Remote, IN",
    "date_posted": "2026-06-24",
    "job_type": "parttime, internship",
    "salary_source": null,
    "interval": null,
    "min_amount": null,
    "max_amount": null,
    "currency": null,
    "is_remote": true,
    "job_function": null,
    "listing_type": null,
    "description": "A Data analyst, gathers, arranges, and records information into computerized databases. Their duties include checking the information included in the documents for accuracy and consistency, and update digital databases and archives. They compile and arrange papers in preparation for processing.\n\n* Data analyst professionals are responsible for handling large am